In [1]:
import pandas as pd
import time
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableParallel
from langchain_core.output_parsers import JsonOutputParser
from langchain.output_parsers import RetryOutputParser
from pydantic import BaseModel, Field
from langchain_aws import ChatBedrockConverse
from typing import List, Dict

In [3]:
word_df = pd.read_csv("intermediate_data/anagram_df.csv")
word_df = word_df.drop("Unnamed: 0", axis=1)

In [4]:
import ast

In [5]:
word_list = [ast.literal_eval(l) for l in word_df.processed_list]

In [6]:
class Answer(BaseModel):
    answer: str
    rationale: str

In [7]:
llm = ChatBedrockConverse(
    #model="anthropic.claude-3-5-haiku-20241022-v1:0",
    model="anthropic.claude-3-5-sonnet-20241022-v2:0",
    temperature=0,
    max_tokens=8192,
)

In [8]:
json_parser = JsonOutputParser(pydantic_object=Answer)
retry_parser = RetryOutputParser.from_llm(parser=json_parser, llm=llm, max_retries=3)

In [9]:
with open("prompt/gmaA_prompt.txt") as f:
    prompt_text = f.read()

In [10]:
format_instructions = json_parser.get_format_instructions()
prompt = PromptTemplate(
    template=prompt_text,
    input_variables="word_list",
    partial_variables={"format_instructions": format_instructions},
)

In [11]:
chain = RunnableParallel(
    completion=prompt | llm, 
    prompt_value=prompt
) | RunnableLambda(lambda x: retry_parser.parse_with_prompt(x["completion"].content, x["prompt_value"]))

In [12]:
import time

In [13]:
response_list = []
start_time = time.time()
for i in range(len(word_list)):
    if i % 5 == 0 and i != 0:
        time_lapse = time.time() - start_time
        print(f"Sleeping at {i}...{time_lapse}")
        if time_lapse < 60:
            time.sleep(60 - (time_lapse))
        start_time = time.time()
    response = chain.invoke({"word_list": word_list[i],})
    response_list.append(response["answer"])

Sleeping at 5...13.843424081802368
Sleeping at 10...13.243861675262451
Sleeping at 15...12.68853235244751


In [14]:
word_df["sonnet35"] = response_list

In [15]:
word_df.to_csv("intermediate_data/gmaA.csv", index=False)